══════════════════════════════════════════════════════════════
 WEEK 12  |  DAY 5  |  ORCHESTRATE — FINAL PIPELINE RUN
══════════════════════════════════════════════════════════════

 LEARNING OBJECTIVES
 ───────────────────
 1. Configure Python logging with timestamp and level
 2. Orchestrate a multi-stage pipeline by chaining outputs
 3. Build a structured run report with timing and error counts

 PIPELINE CONTEXT
 ─────────────────
 This is Stage 5 (final) of the capstone pipeline:
   Day 1: Docker + Kafka setup (simulation)
   Day 2: Extract data from API, produce to Kafka
   Day 3: Store raw records in Bronze layer (MinIO)
   Day 4: Transform and load to PostgreSQL (Silver)
   Day 5: Orchestrate the full pipeline with logging  <-- today

 Today we wire all four stages together into a single
 run_pipeline() function.  Each stage's output becomes the
 next stage's input.  Failures are caught per-stage so one
 broken stage does not crash the entire pipeline.

 TIME:  ~45 minutes

══════════════════════════════════════════════════════════════


In [ ]:


import logging
import time
import os
import json
import sqlite3
from datetime import datetime




 ─────────────────────────────────────────────────────────────
 ARCHITECTURE DECISION
 ─────────────────────
 Choosing between: custom logging + Grafana vs OpenTelemetry vs managed APM (Datadog/New Relic).
 Rule of thumb: use custom metrics + Grafana for full control and no vendor lock-in. Use OpenTelemetry for portable, standards-based traces across microservices. Move to managed APM when the team needs pre-built dashboards and on-call alerting out of the box.

══════════════════════════════════════════════════════════════
 CONCEPT 1 — PYTHON LOGGING MODULE
══════════════════════════════════════════════════════════════

 The logging module is the standard way to emit structured
 messages in Python programs.  Unlike print(), log messages
 include a level (DEBUG, INFO, WARNING, ERROR, CRITICAL) and
 can be routed to files, external systems, or filtered by
 level at runtime.

 basicConfig() sets up the root logger with one call:
   level   — minimum severity to display (INFO = show INFO+)
   format  — format string for each log line
   handlers — where to send log output (default: stderr)

 Useful format tokens:
   %(asctime)s    — timestamp (e.g. 2024-01-15 10:22:05,123)
   %(levelname)s  — INFO, WARNING, ERROR, etc.
   %(message)s    — the message string

 The four main log levels in order of severity:
   logging.debug(msg)    — verbose detail (disabled in INFO mode)
   logging.info(msg)     — normal progress messages
   logging.warning(msg)  — something unexpected but not fatal
   logging.error(msg)    — a stage failed; pipeline may continue

 Call basicConfig only once per program (at the top-level
 script).  Calling it inside a function has no effect if the
 root logger was already configured.

EXAMPLE ──────────────────────────────────────────────────────


Configure the root logger (call this once at the top of the script).


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

logging.info("Logging is configured.")
logging.info("Pipeline script starting.")
logging.warning("This is a WARNING — something unexpected but the script continues.")
logging.error("This is an ERROR — a stage would have failed here.")
logging.info("After the ERROR, the pipeline continues.")




══════════════════════════════════════════════════════════════
 CONCEPT 2 — ORCHESTRATION PATTERN
══════════════════════════════════════════════════════════════

 Orchestration means running each stage in order and passing
 the output of one stage as the input to the next.

 The pattern for a robust pipeline:

   try:
       stage_output = run_stage(previous_output)
       logging.info(f"Stage done: {len(stage_output)} records")
   except Exception as e:
       logging.error(f"Stage failed: {e}")
       stage_output = []   # continue with empty data

 If a stage fails, we set its output to [] and let the next
 stage run.  The downstream stage will produce 0 records,
 which is safe.  The run report will capture the error.

 Why catch Exception (broad) instead of a specific type?
 In orchestration code, we want to catch any unexpected error
 (database down, disk full, network timeout) so the orchestrator
 itself never crashes.  The error is recorded in the report.

EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

def stage_extract_sim(resource_id, limit):
    """Simulate the extract stage — return fake records."""
    records = [
        {"_id": i, "district": f"District_{i % 4 + 1}", "count": i * 10, "year": 2023}
        for i in range(1, limit + 1)
    ]
    return records

def stage_transform_sim(raw_records):
    """Simulate the transform stage — add loaded_at to each record."""
    cleaned = []
    for r in raw_records:
        r_copy = dict(r)
        r_copy["loaded_at"] = datetime.now().isoformat()
        cleaned.append(r_copy)
    return cleaned

def stage_load_sim(records):
    """Simulate the load stage — insert into in-memory SQLite."""
    conn = sqlite3.connect(":memory:")
    c = conn.cursor()
    c.execute("CREATE TABLE pipeline_data (id INTEGER, district TEXT, count INTEGER, year INTEGER, loaded_at TEXT)")
    rows = [(r["_id"], r["district"], r["count"], r["year"], r["loaded_at"]) for r in records]
    c.executemany("INSERT INTO pipeline_data VALUES (?, ?, ?, ?, ?)", rows)
    conn.commit()
    c.execute("SELECT COUNT(*) FROM pipeline_data")
    row_count = c.fetchone()[0]
    conn.close()
    return row_count



Run the three stages in sequence, passing outputs forward.


In [ ]:
logging.info("Orchestration demo starting.")

try:
    raw = stage_extract_sim("demo-resource", limit=10)
    logging.info(f"Stage EXTRACT done: {len(raw)} records fetched")
except Exception as e:
    logging.error(f"Stage EXTRACT failed: {e}")
    raw = []

try:
    cleaned = stage_transform_sim(raw)
    logging.info(f"Stage TRANSFORM done: {len(cleaned)} records cleaned")
except Exception as e:
    logging.error(f"Stage TRANSFORM failed: {e}")
    cleaned = []

try:
    loaded_count = stage_load_sim(cleaned)
    logging.info(f"Stage LOAD done: {loaded_count} rows in database")
except Exception as e:
    logging.error(f"Stage LOAD failed: {e}")
    loaded_count = 0

logging.info("Orchestration demo complete.")




══════════════════════════════════════════════════════════════
 CONCEPT 3 — RUN REPORT
══════════════════════════════════════════════════════════════

 A run report is a dict that captures the outcome of one
 pipeline execution.  It is printed at the end of every run
 so the operator can see at a glance what happened.

 Standard run report keys:

   started_at        — ISO timestamp when run_pipeline() started
   finished_at       — ISO timestamp when run_pipeline() ended
   duration_seconds  — float, finished - started in seconds
   records_fetched   — int, how many records the extract stage got
   records_loaded    — int, how many rows were written to the DB
   errors            — list of error message strings (empty = success)

 Use time.time() to measure elapsed time:
   t_start = time.time()
   ... do work ...
   duration = round(time.time() - t_start, 2)

 After building the report dict, print it one key per line
 for readability.

EXAMPLE ──────────────────────────────────────────────────────


In [ ]:

t0 = time.time()
time.sleep(0.05)   # simulate work taking 50 ms
t1 = time.time()

demo_report = {
    "started_at":       datetime.now().isoformat()[:19],
    "finished_at":      datetime.now().isoformat()[:19],
    "duration_seconds": round(t1 - t0, 2),
    "records_fetched":  10,
    "records_loaded":   10,
    "errors":           [],
}

print("\nRun report example:")
for key, value in demo_report.items():
    print(f"  {key:<22} : {value}")




══════════════════════════════════════════════════════════════
 EXERCISE 1
══════════════════════════════════════════════════════════════

 Set up logging and write a function run_pipeline(resource_id,
 output_folder, db_path) that orchestrates all four stages.

 The stages (all simulated inline — no imports needed):
   Stage 1 EXTRACT  : call stage_extract_sim(resource_id, limit=20)
   Stage 2 PRODUCE  : append each record to a list called topic_queue
   Stage 3 BRONZE   : call json.dumps() on topic_queue, write to file
                      named f"{output_folder}/raw_{today}.json"
                      where today = datetime.now().strftime("%Y-%m-%d")
   Stage 4 LOAD     : call stage_load_sim(topic_queue)

 After each stage, log an INFO message:
   "Stage EXTRACT complete: 20 records"
   "Stage PRODUCE complete: 20 records in queue"
   "Stage BRONZE complete: saved to raw_2024-01-15.json"
   "Stage LOAD complete: 20 rows in database"

 Call run_pipeline("gov-resource-001", output_folder, ":memory:")

 Expected output (logging lines):
   2024-01-15 10:22:05  INFO      Stage EXTRACT complete: 20 records
   2024-01-15 10:22:05  INFO      Stage PRODUCE complete: 20 records in queue
   2024-01-15 10:22:05  INFO      Stage BRONZE complete: saved to raw_2024-01-15.json
   2024-01-15 10:22:05  INFO      Stage LOAD complete: 20 rows in database


--- starting data ---


In [ ]:
output_folder = os.path.join(os.path.dirname(__file__), "pipeline_output")
os.makedirs(output_folder, exist_ok=True)






(write your code here)


══════════════════════════════════════════════════════════════
 EXERCISE 2
══════════════════════════════════════════════════════════════

 Modify run_pipeline from Exercise 1 to add per-stage
 error handling.

 Wrap each stage call in try/except Exception as e:
   - On success: log INFO as before
   - On failure: log ERROR with the message, set stage output to []
     and append the error message string to an errors list
   - The pipeline must always reach Stage 4 even if earlier stages fail

 To test the error path, add this line at the top of run_pipeline
 (before Stage 1):
   raise_on_stage = 2   # set to 1,2,3,4 to simulate a failure

 Then in Stage 2's try block, before the main code, add:
   if raise_on_stage == 2:
       raise RuntimeError("Simulated Stage 2 failure")

 Call run_pipeline with raise_on_stage = 2.

 Expected output:
   INFO      Stage EXTRACT complete: 20 records
   ERROR     Stage PRODUCE failed: Simulated Stage 2 failure
   INFO      Stage BRONZE complete: saved to raw_2024-01-15.json
   INFO      Stage LOAD complete: 0 rows in database
   WARNING   Pipeline completed with 1 error(s)


--- starting data ---
No additional variables needed


(write your code here)


══════════════════════════════════════════════════════════════
 EXERCISE 3
══════════════════════════════════════════════════════════════

 Add a run report to run_pipeline from Exercise 2.

 At the START of run_pipeline, record:
   t_start    = time.time()
   started_at = datetime.now().isoformat()[:19]

 At the END (after all four stages and error checks), build
 a run_report dict with these exact keys:
   started_at        — string from above
   finished_at       — datetime.now().isoformat()[:19]
   duration_seconds  — round(time.time() - t_start, 2)
   records_fetched   — count from Stage 1 (0 if failed)
   records_loaded    — count from Stage 4 (0 if failed)
   errors            — list of error strings (empty = clean run)

 Print the run report one key per line, then return it.

 Call run_pipeline with raise_on_stage = 0 (no failures) and
 print the returned report to verify all keys are present.

 Expected output:
   INFO      Stage EXTRACT complete: 20 records
   INFO      Stage PRODUCE complete: 20 records in queue
   INFO      Stage BRONZE complete: saved to raw_2024-01-15.json
   INFO      Stage LOAD complete: 20 rows in database
   Run report:
     started_at             : 2024-01-15T10:22:05
     finished_at            : 2024-01-15T10:22:05
     duration_seconds       : 0.02
     records_fetched        : 20
     records_loaded         : 20
     errors                 : []


--- starting data ---
No additional variables needed


(write your code here)


## ══════════════════════════════════════════════════════════════
##  CONCEPT 4 — FULL OBSERVABILITY STACK
## ══════════════════════════════════════════════════════════════
Royal Road Standard W12: combine structured logs, run metrics, and cost tracking
into one `PipelineObserver` object — and call `dashboard()` at the end of every run.

An observable pipeline records **three things** for every run:

| Layer | What it captures |
|-------|-----------------|
| Structured logs | every stage name, status (OK / FAILED), duration |
| Run metrics | records in, records out, error count |
| Cost snapshot | API calls made, tokens used, estimated dollar cost |

This pattern is what a Principal AI Solutions Engineer owns in production.
Every alert, SLA, and cost report flows from this object.

**EXAMPLE**

In [ ]:
import logging, datetime

class PipelineObserver:
    def __init__(self, pipeline_name):
        self.name        = pipeline_name
        self.started_at  = datetime.datetime.now()
        self.stages      = []        # list of stage records
        self.api_calls   = 0
        self.tokens_used = 0
        self.cost_per_k  = 0.002     # $ per 1 000 tokens

    def record_stage(self, stage_name, records_in, records_out, error=None):
        entry = {
            "stage":       stage_name,
            "records_in":  records_in,
            "records_out": records_out,
            "ok":          error is None,
            "error":       str(error) if error else None,
        }
        self.stages.append(entry)
        status = "OK" if error is None else f"FAILED ({error})"
        logging.info("[%s] %s: %d->%d | %s",
                     self.name, stage_name, records_in, records_out, status)

    def add_api_call(self, tokens=0):
        self.api_calls   += 1
        self.tokens_used += tokens

    def dashboard(self):
        elapsed = (datetime.datetime.now() - self.started_at).total_seconds()
        failed  = [s for s in self.stages if not s["ok"]]
        cost    = (self.tokens_used / 1000) * self.cost_per_k
        sep     = "=" * 52
        print(sep)
        print(f"  PIPELINE DASHBOARD  {self.name}")
        print(sep)
        print(f"  Duration       : {elapsed:.2f}s")
        print(f"  Stages run     : {len(self.stages)}")
        print(f"  Stages failed  : {len(failed)}")
        print(f"  API calls      : {self.api_calls}")
        print(f"  Tokens used    : {self.tokens_used}")
        print(f"  Est. cost      : ${cost:.4f}")
        if failed:
            print(f"  Failed stages  : {[f['stage'] for f in failed]}")
        print(sep)

# Demo: wire the observer into a 3-stage run
obs = PipelineObserver("gov_data_pipeline")

obs.record_stage("extract",   records_in=0,   records_out=500)
obs.add_api_call(tokens=1200)

obs.record_stage("transform", records_in=500, records_out=487)

obs.record_stage("load",      records_in=487, records_out=487)

obs.dashboard()

## ══════════════════════════════════════════════════════════════
##  EXERCISE 4
## ══════════════════════════════════════════════════════════════
Royal Road Standard W12: wire `PipelineObserver` into a 4-stage run.

Your pipeline has these stages (simulate with `record_stage` / `add_api_call`):

| Stage | records_in | records_out | API call? |
|-------|-----------|-------------|-----------|
| fetch_api | 0 | 300 | yes — 800 tokens |
| validate | 300 | 285 | no |
| transform | 285 | 285 | no |
| load_to_db | 285 | 285 | no |

1. Create a `PipelineObserver` named `"final_pipeline"`.
2. Record each stage in order.
3. Call `add_api_call(tokens=800)` after the fetch stage.
4. Call `dashboard()` at the end.

Expected output:
```
====================================================
  PIPELINE DASHBOARD  final_pipeline
====================================================
  Duration       : 0.00s
  Stages run     : 4
  Stages failed  : 0
  API calls      : 1
  Tokens used    : 800
  Est. cost      : $0.0016
====================================================
```
--- starting data --- (no additional variables needed)

In [ ]:
# Create your PipelineObserver and call record_stage() / add_api_call() below.

## ══════════════════════════════════════════════════════════════
##  CONCEPT 5 — SMART SCHEMA AGENT: STAGE 7 — FASTAPI PRODUCTION WRAPPER
## ══════════════════════════════════════════════════════════════
Stage 7 adds a production HTTP layer to the `SmartSchemaAgent` built across W6–W11.
Five endpoints expose the governed NL→SQL interface to external callers:

| Method | Path | Agent capability used |
|--------|------|-----------------------|
| GET | /health | liveness — is the service up? |
| GET | /schema | Stage 1: table metadata from sqlite_master |
| POST | /query | Stage 5: natural-language → SQL → rows |
| GET | /audit | Stage 2: last N rows from the audit log |
| POST | /validate | Stage 3: safety check before execution |

This is the full agent running as a governed, observable REST API.
In production: replace `DB_PATH = ":memory:"` with your real database path.

**EXAMPLE**

In [ ]:
# SIMULATION MODE
# To run as a real server: uvicorn <this_file>:app --reload
# Replace DB_PATH with your real SQLite file in production.

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import sqlite3

app = FastAPI(title="Smart Schema Agent API", version="7.0")
DB_PATH = ":memory:"

DANGEROUS = {"DROP", "TRUNCATE", "DELETE"}


def _run_sql(sql, limit=50):
    con = sqlite3.connect(DB_PATH)
    try:
        cur = con.execute(sql)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchmany(limit)
        return [dict(zip(cols, row)) for row in rows]
    finally:
        con.close()


class QueryRequest(BaseModel):
    question: str
    max_rows: int = 10

class ValidateRequest(BaseModel):
    sql: str


@app.get("/health")
def health():
    return {"status": "ok", "agent": "SmartSchemaAgent", "version": "7.0"}


@app.get("/schema")
def schema():
    con = sqlite3.connect(DB_PATH)
    rows = con.execute(
        "SELECT name, sql FROM sqlite_master WHERE type='table'"
    ).fetchall()
    con.close()
    return {r[0]: r[1] for r in rows}


@app.post("/query")
def query(req: QueryRequest):
    # Stage 5: in production, pass req.question to the LangChain NL->SQL agent
    # Here we simulate with a fixed query so no API key is required
    simulated_sql = f"SELECT * FROM audit_log LIMIT {req.max_rows}"
    try:
        rows = _run_sql(simulated_sql, req.max_rows)
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc))
    return {"sql": simulated_sql, "rows": rows, "row_count": len(rows)}


@app.get("/audit")
def audit(limit: int = 20):
    try:
        return _run_sql(
            f"SELECT * FROM audit_log ORDER BY rowid DESC LIMIT {limit}"
        )
    except Exception:
        return []


@app.post("/validate")
def validate(req: ValidateRequest):
    upper = req.sql.upper()
    for kw in DANGEROUS:
        if kw in upper:
            return {"safe": False, "reason": "destructive operation"}
    return {"safe": True, "reason": "ok"}


# Print endpoint map (no server needed to see this output)
endpoints = [
    ("GET ", "/health",   "service liveness"),
    ("GET ", "/schema",   "table metadata (Stage 1)"),
    ("POST", "/query",    "NL question -> SQL + rows (Stage 5)"),
    ("GET ", "/audit",    "last N audit entries (Stage 2)"),
    ("POST", "/validate", "SQL safety check (Stage 3)"),
]
print("SmartSchemaAgent API — Stage 7")
for method, path, desc in endpoints:
    print(f"  {method}  {path:12}  {desc}")
print()
print("To run:  uvicorn <filename>:app --reload")

## ══════════════════════════════════════════════════════════════
##  EXERCISE 5 — SMART SCHEMA AGENT: STAGE 7
## ══════════════════════════════════════════════════════════════
Add a `GET /stats` endpoint to the app defined above.

The endpoint must return a JSON object with exactly three keys:

| Key | Value |
|-----|-------|
| `"tables"` | number of tables in the schema (from `sqlite_master`) |
| `"audit_entries"` | total rows in `audit_log` (return 0 if the table does not exist) |
| `"agent_version"` | the string `"7.0"` |

Expected response when the DB has 2 tables and 5 audit rows:
```json
{
  "tables": 2,
  "audit_entries": 5,
  "agent_version": "7.0"
}
```

--- starting data ---

In [ ]:
# Add your @app.get("/stats") endpoint below.
# Use _run_sql() for database queries; wrap audit_log access in try/except.